# Visualize example ligands in 3D with atom names

In [ ]:
!pip install -q rdkit py3Dmol

import collections
import py3Dmol
from rdkit import Chem as rd_chem
from rdkit.Chem import AllChem as rd_all_chem

## Example SMILES strings from manifests

In [ ]:
EXAMPLE_SMILES = {
    'serine_esterase/es': 'CC(=O)Oc2ccc1c(C)cc(=O)oc1c2',
    'serine_esterase/complex': 'Cc1cc(=O)oc2cc(O[C@@H](C)[O-])ccc12',
    'serine_esterase/aei': 'CC=O',
    'serine_esterase/ti2': 'C[C@H](O)[O-]',
    'dehp_esterase/es': 'CCCC[C@H](CC)COC(=O)c1ccccc1C(=O)OC[C@@H](CC)CCCC',
    'dehp_esterase/complex': (
        'CCCC[C@H](CC)COC(=O)c1ccccc1[C@@H]([O-])OC[C@@H](CC)CCCC'
    ),
    'dehp_esterase/aei': 'CCCC[C@@H](CC)COC(=O)c1ccccc1C=O',
    'dehp_esterase/ti2': 'CCCC[C@@H](CC)COC(=O)c1ccccc1[C@@H](O)[O-]',
    'nitrene_transferase/heme_substrate': 'NCCCCCC1=CC=CC=C1',
    'nitrene_transferase/heme_piperidine_R': 'C1CC[C@H](NC1)C1=CC=CC=C1',
    'nitrene_transferase/heme_pyrrolidine_S': 'C([C@@H]1CCCN1)C1=CC=CC=C1',
    'nitrene_transferase/fiveazidopentylbenzene_heme_1': (
        'CC1=C(CCC(O)=O)C2=NC1=Cc1c(C=C)c(C)c3\\C=C4/N=C(/C=C5\\N([Fe](N(CCCCCC6=CC=CC=C6)N=N)n13)\\C(=C/2)C(CCC(O)=O)=C5C)C(C)=C4C=C'
    ),
    'nitrene_transferase/fiveazidopentylbenzene_heme_2': (
        '[Fe]N(CCCCCC1=CC=CC=C1)N=N.CC1=C(CCC([O-])=O)/C2=C/C3=N/C(=C\\C4=C(C)C(C=C)=C([N-]4)/C=C4\\N=C(\\C=C\\1/[N-]\\2)C(C=C)=C4C)/C(C)=C3CCC([O-])=O'
    ),
}

## 3D visualization with atom names

In [ ]:
def assign_atom_names_from_graph(mol: rd_chem.Mol) -> rd_chem.Mol:
  mol = rd_chem.Mol(mol)
  element_counts = collections.Counter()
  for atom in mol.GetAtoms():
    element = atom.GetSymbol()
    element_counts[element] += 1
    atom.SetProp('atom_name', f'{element.upper()}{element_counts[element]}')
  return mol


def get_random_conformer(
    mol: rd_chem.Mol, random_seed: int
) -> rd_chem.Conformer:
  params = rd_all_chem.ETKDGv3()
  params.randomSeed = random_seed
  mol_copy = rd_chem.Mol(mol)
  conformer_id = rd_all_chem.EmbedMolecule(mol_copy, params)
  return mol_copy.GetConformer(conformer_id)


def visualize_smiles_3d(smiles: str, seed: int = 0) -> py3Dmol.view:
  mol = rd_chem.MolFromSmiles(smiles)
  mol = rd_chem.AddHs(mol)
  mol = assign_atom_names_from_graph(mol)
  conformer = get_random_conformer(mol=mol, random_seed=seed)
  mol.AddConformer(conformer)
  mol = rd_chem.RemoveHs(mol)

  view = py3Dmol.view(width=600, height=600)
  view.addModel(rd_chem.MolToMolBlock(mol), 'sdf')
  view.setStyle({'stick': {}, 'sphere': {'scale': 0.25}})

  positions = mol.GetConformer().GetPositions()
  for atom, (x, y, z) in zip(mol.GetAtoms(), positions):
    view.addLabel(
        atom.GetProp('atom_name'),
        {'position': {'x': float(x), 'y': float(y), 'z': float(z)}},
    )

  view.zoomTo()
  return view


visualize_smiles_3d(EXAMPLE_SMILES['serine_esterase/complex'])